In [0]:
%run ./engine

In [0]:
%run ./omop_config

In [0]:
cfg = OMOP_CONFIG
PROFILER = cfg.profiler
RARE = cfg.rare
VALPROT_OVERRIDE = cfg.valprot_override
VOLUME = cfg.target["volume"]
CONFIG_DIR = f"{VOLUME}/config"
REPORT_DIR = f"{VOLUME}/reports"
DP = cfg.dp
MODEL_LIMITS = cfg.model_limits
PII_SCRUB = cfg.pii

In [0]:
# =====================================================================================
# §1 TRAIN-CONFIG SMOKE
# =====================================================================================
import json, os, traceback
_log, _fail = [], []
def out(s): _log.append(str(s))
def check(name, cond):
    out(("  PASS  " if cond else "  FAIL  ") + name)
    if not cond: _fail.append(name)

def validate_sdk_contract(t):
    fk_cols = [fk["column"] for fk in t.get("foreign_keys") or []]
    pk = t.get("primary_key")
    if pk and pk in fk_cols:
        return False, f"pk {pk} also declared as fk"
    cols = t.get("columns")
    if cols is not None:
        names = {c["name"] for c in cols}
        if pk and pk not in names:
            return False, f"pk {pk} missing from columns"
        for c in fk_cols:
            if c not in names:
                return False, f"fk {c} missing from columns"
    return True, ""

try:
    sdtypes = load_sdtypes()
    check("sdtypes seed non-empty", len(sdtypes) >= 10)

    out("== reference generator ==")
    ref_frames = prepare_frames("reference")
    ref_tables = build_generator_tables("reference", sdtypes, ref_frames)
    ref_by_name = {t["name"]: t for t in ref_tables}
    check("reference has 3 tables", len(ref_tables) == 3)
    check("care_site.location_id NOT trained",
          all(c["name"] != "location_id" for c in ref_by_name["care_site"].get("columns", [])))
    check("provider.care_site_id NOT trained",
          all(c["name"] != "care_site_id" for c in ref_by_name["provider"].get("columns", [])))
    check("reference no DP",
          all("differential_privacy" not in t.get("tabular_model_configuration", {}) for t in ref_tables))
    check("reference no value_protection override (keeps SDK default)",
          all("value_protection" not in t.get("tabular_model_configuration", {}) for t in ref_tables))
    check("location PII scrubbed from frame",
          not any(c in ref_frames["location"].columns for c in PII_SCRUB["location"]))
    check("location keeps IMD_Quintile", "IMD_Quintile" in ref_frames["location"].columns)
    # R4a: provider/care_site identifiers are PII -> dropped from the training frame.
    check("provider PII (name/npi/dea) scrubbed from frame",
          not any(c in ref_frames["provider"].columns for c in PII_SCRUB.get("provider", [])))
    check("care_site PII (name) scrubbed from frame",
          not any(c in ref_frames["care_site"].columns for c in PII_SCRUB.get("care_site", [])))
    for t in ref_tables:
        ok, why = validate_sdk_contract(t)
        check(f"sdk_contract {t['name']}", ok)
        if not ok: out("    " + why)

    out("== core generator ==")
    core_frames = prepare_frames("core")
    core_tables = build_generator_tables("core", sdtypes, core_frames)
    core_by_name = {t["name"]: t for t in core_tables}
    check("core has 14 tables", len(core_tables) == 14)

    check("person is subject", "foreign_keys" not in core_by_name["person"])
    vfk = core_by_name["visit_occurrence"]["foreign_keys"][0]
    check("visit context fk -> person",
          vfk == {"column": "person_id", "referenced_table": "person", "is_context": True})

    for t in ["measurement", "drug_exposure", "condition_occurrence",
              "procedure_occurrence", "device_exposure", "observation_visit"]:
        cfk = core_by_name[t]["foreign_keys"][0]
        check(f"{t} context fk -> visit_occurrence",
              cfk == {"column": "visit_occurrence_id",
                      "referenced_table": "visit_occurrence", "is_context": True})
        check(f"{t} person_id dropped from frame", "person_id" not in core_frames[t].columns)
    check("measurement visit_detail_id dropped", "visit_detail_id" not in core_frames["measurement"].columns)
    check("measurement event-link dropped", "measurement_event_id" not in core_frames["measurement"].columns)

    def _tmc(t): return t.get("tabular_model_configuration", {})
    check("max_training_time capped on every core table",
          all(_tmc(t).get("max_training_time") == MODEL_LIMITS["max_training_time"] for t in core_tables))
    check("max_epochs capped on every core table",
          all(_tmc(t).get("max_epochs") == MODEL_LIMITS["max_epochs"] for t in core_tables))
    check("max_training_time is bounded (< 14400 default)",
          MODEL_LIMITS["max_training_time"] < 14400)

    p = f"{CONFIG_DIR}/rarity_profile.json"
    have_profile = os.path.exists(p)
    profile = json.load(open(p)) if have_profile else {}
    _cohort_mode = profile.get("header", {}).get("cohort_mode") if have_profile else None
    fresh = profile_is_fresh(profile, RARITY_CONFIG_HASH(), _cohort_mode) if have_profile else False
    check("rarity profile present", have_profile)
    if have_profile:
        check("rarity profile fresh (config_hash + mode match)", fresh)
    policies = resolve_all_policies(topo_order(cfg, "core"), profile.get("tables", {}),
                                    VALPROT_OVERRIDE, legacy_valprot_off(), RARE["seqlen_support_min"])
    def _apply(t):
        pol = policies[t["name"]]
        tmc = dict(t.get("tabular_model_configuration", {}))
        if pol["value_protection"]:
            tmc.pop("value_protection", None)
        else:
            tmc["value_protection"] = False
        return tmc
    applied = {t["name"]: _apply(t) for t in core_tables}
    vp_off = {name for name, tmc in applied.items() if tmc.get("value_protection") is False}
    resolved_off = {t for t in topo_order(cfg, "core") if not policies[t]["value_protection"]}
    check("applied vp_off == resolved policy", vp_off == resolved_off)
    if have_profile and fresh:
        check("resolved vp_off == legacy 5 (death override ON)", vp_off == set(legacy_valprot_off()))
    check("death pinned value_protection ON (override)", "death" not in vp_off)
    check("legacy 5 all resolve value_protection OFF",
          set(legacy_valprot_off()) <= vp_off or not (have_profile and fresh))

    check("DP on every core table",
          all("differential_privacy" in t.get("tabular_model_configuration", {}) for t in core_tables))
    def _dp(t): return t["tabular_model_configuration"]["differential_privacy"]
    check("value_protection_epsilon set on every core table",
          all("value_protection_epsilon" in _dp(t) for t in core_tables))
    vp_on = [t for t in topo_order(cfg, "core") if t not in vp_off]
    ce = composed_epsilon(vp_on)
    check("composed eps = train(all) + valprot(on) + profiler",
          abs(ce["total"] - (DP["max_epsilon"]*len(core_tables)
                             + DP["value_protection_epsilon"]*len(vp_on)
                             + PROFILER["epsilon_profiler"])) < 1e-9)
    check("composed eps includes profiler term (> 0)", ce["profiler"] == PROFILER["epsilon_profiler"])
    out(f"  resolved vp_off={sorted(vp_off)} | composed eps={ce['total']:.3f} "
        f"(train {ce['train']:.1f} + valprot {ce['valprot']:.1f} over {len(vp_on)} + profiler {ce['profiler']:.1f})")

    menc = {c["name"]: c["model_encoding_type"] for c in core_by_name["measurement"]["columns"]}
    check("measurement_concept_id -> categorical", menc.get("measurement_concept_id") == "TABULAR_CATEGORICAL")
    check("value_as_number -> numeric", menc.get("value_as_number") == "TABULAR_NUMERIC_AUTO")
    check("measurement_date -> datetime", menc.get("measurement_date") == "TABULAR_DATETIME")
    check("measurement pk in columns (AUTO)", menc.get("measurement_id") == "AUTO")
    check("measurement context fk in columns (AUTO)", menc.get("visit_occurrence_id") == "AUTO")

    death = core_by_name["death"]
    check("death has no primary_key", "primary_key" not in death)
    check("death context fk -> person",
          death.get("foreign_keys", [{}])[0].get("column") == "person_id")

    for t in core_tables:
        ok, why = validate_sdk_contract(t)
        check(f"sdk_contract {t['name']}", ok)
        if not ok: out("    " + why)

    person_pks = set(core_frames["person"]["person_id"].dropna().tolist())
    visit_pks = set(core_frames["visit_occurrence"]["visit_occurrence_id"].dropna().tolist())
    for t in ["measurement", "drug_exposure", "condition_occurrence",
              "procedure_occurrence", "device_exposure", "observation_visit"]:
        fkv = core_frames[t]["visit_occurrence_id"].dropna()
        check(f"{t} no orphan visit fk in training input", int((~fkv.isin(visit_pks)).sum()) == 0)
    for t in ["visit_occurrence", "observation_period", "condition_era", "drug_era", "dose_era",
              "observation_person", "death"]:
        fkv = core_frames[t]["person_id"].dropna()
        check(f"{t} no orphan person fk in training input", int((~fkv.isin(person_pks)).sum()) == 0)

except Exception as e:
    out("EXCEPTION: " + repr(e)); out(traceback.format_exc()); _fail.append("EXCEPTION")

summary1 = "PASS: train config smoke OK" if not _fail else f"FAIL: {_fail}"
with open(f"{REPORT_DIR}/smoke_train_config.log", "w") as f:
    f.write("\n".join(_log) + "\n\n" + summary1 + "\n")
print("\n".join(_log)); print(summary1)

In [0]:
# =====================================================================================
# §2 C1 POLICY DIFF — profile-driven vs legacy OFF sets
# =====================================================================================
import json as _json2, os as _os2
p = f"{CONFIG_DIR}/rarity_profile.json"
prof = _json2.load(open(p)) if _os2.path.exists(p) else {}
fresh2 = profile_is_fresh(prof, RARITY_CONFIG_HASH(), "full")
ptables = prof.get("tables", {})

core = topo_order(cfg, "core")
pol_profile = {t: collapse_policy(t, ptables, {}, [], RARE["seqlen_support_min"]) for t in core}
pol_real = resolve_all_policies(core, ptables, VALPROT_OVERRIDE, legacy_valprot_off(),
                                RARE["seqlen_support_min"])

legacy_off = set(legacy_valprot_off())
profile_off = {t for t in core if not pol_profile[t]["value_protection"]}
real_off = {t for t in core if not pol_real[t]["value_protection"]}

lines = []
def L(s): lines.append(str(s)); print(s)

L(f"profile fresh={fresh2} config_hash={RARITY_CONFIG_HASH()} cohort_mode={prof.get('header',{}).get('cohort_mode')}")
L(f"legacy value_protection-OFF set ({len(legacy_off)}): {sorted(legacy_off)}")
L(f"PROFILE-driven OFF set     ({len(profile_off)}): {sorted(profile_off)}")
L(f"RESOLVED (real) OFF set    ({len(real_off)}): {sorted(real_off)}")
L("")
L("per-table detail (table | profile vp | source | best nonzero support | n_parents):")
for t in core:
    ent = ptables.get(t, {})
    sl = ent.get("seqlen")
    best, npar = None, None
    if sl:
        npar = sl.get("n_parents", 0)
        bins, counts = sl.get("bins", []), sl.get("bin_counts", [])
        best = 0.0
        for i, c in enumerate(counts):
            if i < len(bins) and bins[i] >= 1 and npar:
                best = max(best, c / npar)
    L(f"  {t:24s} | vp={str(pol_profile[t]['value_protection']):5s} | {pol_profile[t]['source']:14s} | "
      f"best={best if best is None else round(best,4)} | n_parents={npar}")

L("")
agree = profile_off == legacy_off
L(f"PROFILE-driven OFF == legacy OFF ? {agree}")
if not agree:
    L(f"  tables PROFILE says off but legacy keeps on: {sorted(profile_off - legacy_off)}")
    L(f"  tables legacy says off but PROFILE keeps on: {sorted(legacy_off - profile_off)}")

with open(f"{REPORT_DIR}/check_c1_policy.log", "w") as f:
    f.write("\n".join(lines) + "\n")

In [0]:
# =====================================================================================
# §3 INTROSPECT SMOKE — live source schemas -> column roles
# =====================================================================================
_ilog, _ifail = [], []
def iout(s): _ilog.append(str(s))
def icheck(name, cond):
    iout(("  PASS  " if cond else "  FAIL  ") + name)
    if not cond:
        _ifail.append(name)

try:
    roles = introspect_roles(cfg, spark, load_sdtypes=load_sdtypes)

    iout("== coverage ==")
    empty = [t for t, r in roles.items() if not r]
    icheck(f"all 17 declared tables resolved (no empty role maps): {empty}", empty == [])
    icheck("roles keyed by every cfg table", set(roles) == set(cfg.tables))

    iout("== concept_id signal ==")
    m_roles = roles.get("measurement", {})
    concept_cols = [c for c in m_roles if c.endswith("_concept_id")]
    icheck("measurement has concept_id columns", len(concept_cols) > 0)
    icheck("every measurement *_concept_id -> categorical",
           all(m_roles[c] == "TABULAR_CATEGORICAL" for c in concept_cols))

    iout("== datetime signal ==")
    if "measurement_date" in m_roles:
        icheck("measurement_date -> datetime (or categorical if no sdtype)",
               m_roles["measurement_date"] in ("TABULAR_DATETIME", "AUTO", "TABULAR_CATEGORICAL"))
    any_dt = any(r == "TABULAR_DATETIME" for tbl in roles.values() for r in tbl.values())
    icheck("at least one column resolved to TABULAR_DATETIME from sdtypes seed", any_dt)

    iout("== table split ==")
    op = roles.get("observation_person", {})
    ov = roles.get("observation_visit", {})
    icheck("observation_person resolved from source_table observation (non-empty)", len(op) > 0)
    icheck("observation_visit resolved from source_table observation (non-empty)", len(ov) > 0)
    icheck("both observation slices share the same source columns", set(op) == set(ov))
    icheck("observation slice has observation_concept_id -> categorical",
           op.get("observation_concept_id") == "TABULAR_CATEGORICAL")

    iout("== pii scrub (the privacy invariant, not the introspect role) ==")
    loc = roles.get("location", {})
    sv = [c for c in loc if c.endswith("_source_value")]
    icheck("location source_value cols -> categorical (if any)",
           all(loc[c] == "TABULAR_CATEGORICAL" for c in sv))
    # R4a: the PRIVACY guarantee is that declared PII is DROPPED FROM TRAINING (trained_columns),
    # regardless of what introspect's source-schema heuristic would assign it. (introspect is
    # advisory over the raw schema; provider_name still resolves categorical via its sdtypes seed
    # entry, but it never reaches the model and is nulled at write time.)
    for tname, cols in cfg.pii.items():
        if tname not in cfg.tables:
            continue
        all_cols = [f.name for f in spark.table(
            f"{cfg.source['catalog']}.{cfg.source['schema']}."
            f"{cfg.tables[tname].source_table or tname}").schema.fields]
        tr = set(trained_columns(cfg, tname, all_cols))
        present = [c for c in cols if c in all_cols]
        icheck(f"{tname} PII dropped from training frame: {present}",
               not (set(present) & tr))

    iout("== ambiguous (AUTO) summary ==")
    for t in topo_order(cfg):
        amb = ambiguous_cols(roles.get(t, {}))
        iout(f"  {t}: {len(amb)} AUTO of {len(roles.get(t, {}))} -> {amb[:8]}")

except Exception as e:
    iout("EXCEPTION: " + repr(e)); iout(traceback.format_exc()); _ifail.append("EXCEPTION")

summary3 = "PASS: introspect smoke OK" if not _ifail else f"FAIL: {_ifail}"
with open(f"{REPORT_DIR}/smoke_introspect.log", "w") as f:
    f.write("\n".join(_ilog) + "\n\n" + summary3 + "\n")
print("\n".join(_ilog)); print(summary3)

In [0]:
# RAISE on any failure so the notebook job status is FAILED.
overall_fail = _fail + _ifail
overall = ("PASS: gates OK" if not overall_fail else f"FAIL: {overall_fail}")
_payload = _json2.dumps({"train_config_smoke": summary1, "c1_fresh": fresh2,
                         "c1_resolved_off": sorted(real_off), "c1_legacy_off": sorted(legacy_off),
                         "introspect_smoke": summary3, "overall": overall})
if overall_fail:
    raise AssertionError(overall + " — see reports smoke_train_config.log / smoke_introspect.log")
dbutils.notebook.exit(_payload)